# Notebook 14 - Experiment 22: Fairness Drift Across Datasets
### Novelty 5
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Nearly every paper in this space computes fairness on one dataset and then talks as
if the result holds everywhere. I wanted to actually test that assumption instead of
inheriting it.

I define Fairness Drift as the absolute difference in a fairness metric between two
datasets, for the same model and the same subgroup comparison. If a model's drift is
low, its fairness profile is basically a property of the model. If the drift is high
on particular subgroups, then its ethical-risk rating only really applies to the
dataset it was measured on — which is a much weaker claim than most papers make.

I test two pairs: TweetEval to SemEval, and TweetEval to Sentiment140. I added the
Sentiment140 one because it was promised in my proposal but had gone missing from the
original experiment plan. From the EDA, TweetEval and SemEval only share about 40% of
their top-500 vocabulary, so some drop is expected — the real question is whether it
lands evenly or piles onto the informal subgroups.

Fills **Table 11**.

## Cell 1: Setup

In [ ]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
!pip install -q aif360 fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 8.6 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 9.4 MB/s eta 0:00:00


## Cell 2: Load Model and Target Datasets

Hyperparameters stay fixed from the source dataset. No retuning on the target —
an organisation deploying a model to a new data source does not retrain it first,
so retuning would measure something other than deployment robustness.

In [ ]:
D = PATHS["data"]

with open(D / "final_model_config.json") as f:
    cfg = json.load(f)
print(f"Model: {cfg['model']}   Features: {cfg['feature_set']}\n")

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")
se_test  = pd.read_parquet(D / "se_test.parquet")

try:
    s140_test = pd.read_parquet(D / "s140_test.parquet")
    HAS_S140 = True
    print(f"Sentiment140 test: {len(s140_test):,} rows")
except FileNotFoundError:
    HAS_S140 = False
    print("Sentiment140 test not found — TweetEval to SemEval only")

print(f"TweetEval test  : {len(tw_test):,}")
print(f"SemEval test    : {len(se_test):,}")

Model: LogisticRegression   Features: Hybrid feature set

Sentiment140 test: 10,000 rows
TweetEval test  : 12,284
SemEval test    : 1,758


## Cell 3: Source Fairness Audit

The baseline. Everything else is measured as a departure from these values.

In [ ]:
pred_source = load_predictions("exp10", "FinalModel")
audit_source = full_fairness_audit(pred_source)

print("SOURCE — TweetEval")
print("="*70)
print(audit_source[["Subgroup", "AIF360 SPD", "AIF360 DIR", "AIF360 AOD"]]
      .round(4).to_string(index=False))

pip install 'aif360[inFairness]'


SOURCE — TweetEval
   Subgroup  AIF360 SPD  AIF360 DIR  AIF360 AOD
emoji-heavy      0.0287      1.0484     -0.0128
slang-heavy      0.0246      1.0416      0.0052
    sarcasm      0.1937      1.3271      0.0666


## Cell 4: Combination One — TweetEval to SemEval

The vectoriser is fitted on TweetEval and applied to SemEval. SemEval words that
never appear in TweetEval are simply dropped, which is exactly what happens when a
deployed model meets unfamiliar vocabulary.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

model = load_model("exp16", "CrossDatasetModel")
with open(D / "crossdataset_vectorizer.pkl", "rb") as f:
    vec = pickle.load(f)

print("Applying TweetEval-trained model to SemEval...\n")

X_se = vec.transform(se_test["text_clean"])
y_se = se_test["sentiment"].values
sg_se = se_test["subgroup_primary"].values

if cfg["model"] == "XGBoost":
    le = LabelEncoder().fit(tw_train["sentiment"].values)
    pred_se = le.inverse_transform(model.predict(X_se))
else:
    pred_se = model.predict(X_se)
proba_se = model.predict_proba(X_se)

save_predictions("exp22", "FinalModel", y_se, pred_se, proba_se,
                 sg_se, dataset="SemEval")

pred_target_se = load_predictions("exp22", "FinalModel", "SemEval")
audit_se = full_fairness_audit(pred_target_se)

print("TARGET — SemEval-2014")
print("="*70)
if not audit_se.empty:
    print(audit_se[["Subgroup", "AIF360 SPD", "AIF360 DIR", "AIF360 AOD"]]
          .round(4).to_string(index=False))
else:
    print("No subgroup comparisons possible — SemEval informal subgroups too small.")

oov = (X_se.sum(axis=1) == 0).sum()
print(f"\nSemEval documents with zero known tokens: {oov}")

Applying TweetEval-trained model to SemEval...

  saved predictions -> exp22_FinalModel_SemEval.parquet  (1,758 rows)
TARGET — SemEval-2014
   Subgroup  AIF360 SPD  AIF360 DIR  AIF360 AOD
slang-heavy     -0.6077      0.0000         NaN
    sarcasm      0.1066      1.1754     -0.0901

SemEval documents with zero known tokens: 0


## Cell 5: Combination Two — TweetEval to Sentiment140

Binary track, because Sentiment140 has no neutral class. Only positive and
negative rows from the source model's output are comparable.

In [ ]:
if HAS_S140:
    print("Applying TweetEval-trained model to Sentiment140...\n")

    X_s140 = vec.transform(s140_test["text_clean"])
    y_s140 = s140_test["sentiment"].values
    sg_s140 = s140_test["subgroup_primary"].values

    if cfg["model"] == "XGBoost":
        pred_s140 = le.inverse_transform(model.predict(X_s140))
    else:
        pred_s140 = model.predict(X_s140)
    proba_s140 = model.predict_proba(X_s140)

    # Sentiment140 is binary — drop neutral predictions for a fair comparison
    mask = pred_s140 != "neutral"
    print(f"  Predictions that were neutral (not possible in S140): "
          f"{(~mask).sum():,} of {len(mask):,}")

    save_predictions("exp22", "FinalModel",
                     y_s140[mask], pred_s140[mask], proba_s140[mask],
                     sg_s140[mask], dataset="Sentiment140")

    pred_target_s140 = load_predictions("exp22", "FinalModel", "Sentiment140")
    audit_s140 = full_fairness_audit(pred_target_s140)

    print("\nTARGET — Sentiment140")
    print("="*70)
    if not audit_s140.empty:
        print(audit_s140[["Subgroup", "AIF360 SPD", "AIF360 DIR", "AIF360 AOD"]]
              .round(4).to_string(index=False))
else:
    audit_s140 = pd.DataFrame()
    print("Skipped — Sentiment140 test split not available.")

Applying TweetEval-trained model to Sentiment140...

  Predictions that were neutral (not possible in S140): 3,765 of 10,000
  saved predictions -> exp22_FinalModel_Sentiment140.parquet  (6,235 rows)

TARGET — Sentiment140
   Subgroup  AIF360 SPD  AIF360 DIR  AIF360 AOD
emoji-heavy      0.3450      1.5268         0.0
slang-heavy      0.0676      1.1032         0.0
    sarcasm      0.1143      1.1744         0.0


## Cell 6: Compute Fairness Drift

In [ ]:
drift_frames = []

if not audit_se.empty:
    d1 = fairness_drift(audit_source, audit_se, "TweetEval", "SemEval-2014")
    d1["Track"] = "A"
    drift_frames.append(d1)

if not audit_s140.empty:
    d2 = fairness_drift(audit_source, audit_s140, "TweetEval", "Sentiment140")
    d2["Track"] = "B"
    drift_frames.append(d2)

if drift_frames:
    drift = pd.concat(drift_frames, ignore_index=True)
    print("="*100)
    print("FAIRNESS DRIFT")
    print("="*100)
    print(drift[["Train Dataset", "Test Dataset", "Track", "Subgroup",
                 "Source AOD", "Target AOD", "Fairness Drift",
                 "Drift > 0.10?"]].round(4).to_string(index=False))

    mean_drift = drift["Fairness Drift"].mean()
    print(f"\nMean Fairness Drift : {mean_drift:.4f}")
    print(f"Threshold           : {DRIFT_THRESHOLD}")

    high = drift[drift["Fairness Drift"] > DRIFT_THRESHOLD]
    if len(high):
        print(f"\n{len(high)} comparison(s) exceed the threshold:")
        for _, r in high.iterrows():
            print(f"  {r['Subgroup']} on {r['Test Dataset']}: "
                  f"drift {r['Fairness Drift']:.4f}")
        print("\nThis model is DOMAIN SENSITIVE. Its fairness profile does not")
        print("transfer. The ethical risk assessment is only valid for the")
        print("dataset it was measured on.")
    else:
        print("\nThis model is FAIRNESS STABLE across the datasets tested.")
        print("Its fairness properties appear intrinsic rather than domain-specific.")
else:
    drift = pd.DataFrame()
    print("No drift computed — no target audits available.")

FAIRNESS DRIFT
Train Dataset Test Dataset Track    Subgroup  Source AOD  Target AOD  Fairness Drift Drift > 0.10?
    TweetEval SemEval-2014     A slang-heavy      0.0052         NaN             NaN            No
    TweetEval SemEval-2014     A     sarcasm      0.0666     -0.0901          0.1567           Yes
    TweetEval Sentiment140     B emoji-heavy     -0.0128      0.0000          0.0128            No
    TweetEval Sentiment140     B slang-heavy      0.0052      0.0000          0.0052            No
    TweetEval Sentiment140     B     sarcasm      0.0666      0.0000          0.0666            No

Mean Fairness Drift : 0.0603
Threshold           : 0.1

1 comparison(s) exceed the threshold:
  sarcasm on SemEval-2014: drift 0.1567

This model is DOMAIN SENSITIVE. Its fairness profile does not
transfer. The ethical risk assessment is only valid for the
dataset it was measured on.


## Cell 7: Table 11

In [ ]:
if not drift.empty:
    cols = ["Train Dataset", "Test Dataset", "Track", "Subgroup",
            "Source AOD", "Target AOD", "Fairness Drift",
            "Source DIR", "Target DIR", "DIR Drift", "Drift > 0.10?",
            "Fairness-Stable or Domain-Sensitive?"]
    table11 = drift[[c_ for c_ in cols if c_ in drift.columns]]

    print("="*110)
    print("TABLE 11 — FAIRNESS DRIFT ACROSS DATASETS")
    print("="*110)
    print(table11.round(4).to_string(index=False))

    mean_drift = table11["Fairness Drift"].mean()
    gen_score  = score_generalisation_risk(mean_drift)
    print(f"\nGeneralisation Risk score: {gen_score} / 3")

    save_result_table(table11, "Table11_Fairness_Drift")
    print("\nNovelty 5 complete. Feeds the Generalisation Risk dimension of Table 6.")
else:
    print("Table 11 not produced.")

TABLE 11 — FAIRNESS DRIFT ACROSS DATASETS
Train Dataset Test Dataset Track    Subgroup  Source AOD  Target AOD  Fairness Drift  Source DIR  Target DIR  DIR Drift Drift > 0.10? Fairness-Stable or Domain-Sensitive?
    TweetEval SemEval-2014     A slang-heavy      0.0052         NaN             NaN      1.0416      0.0000     1.0416            No                      Fairness-stable
    TweetEval SemEval-2014     A     sarcasm      0.0666     -0.0901          0.1567      1.3271      1.1754     0.1517           Yes                     Domain-sensitive
    TweetEval Sentiment140     B emoji-heavy     -0.0128      0.0000          0.0128      1.0484      1.5268     0.4784            No                      Fairness-stable
    TweetEval Sentiment140     B slang-heavy      0.0052      0.0000          0.0052      1.0416      1.1032     0.0616            No                      Fairness-stable
    TweetEval Sentiment140     B     sarcasm      0.0666      0.0000          0.0666      1.3271      1